
# Bengali NLP Safety — V6: official metric correction + conservative ensemble

This notebook reuses the exact probabilities from the supplied V5 result bundle. **No GPU training is required.**

### Why V6 exists
V5 evaluated candidates with an equal macro-average across all seven positive-class F1 scores. The competition's published metric is different:

- `S_A = Macro-F1(is_toxic)` — macro F1 across the toxic/non-toxic classes.
- `S_B = mean(F1)` across the six toxicity attributes.
- `Final = 0.5 * S_A + 0.5 * S_B`.

Because `is_toxic` receives half of the whole competition score, the V5 threshold `0.41` was not optimized for the leaderboard objective. Re-optimizing the existing probabilities produces a materially different threshold near `0.665`.

V6 also tests a deliberately small BanglaBERT epoch-4 blend. The robust choice found from grouped five-fold threshold checks is:

- 5% BanglaBERT epoch 4 for `is_toxic`, `implicit_toxicity`, `contains_profanity`, `threatening`, and `identity_attack`.
- 0% for `sarcasm` and `sexual_content` because those labels did not show a reliable cross-fold gain from the blend.
- 95–100% of the proven V2 probabilities remain the foundation.

The notebook generates two submissions. **`submission_v6_recommended.csv` is the first one to submit.** `submission_v6_metric_fix.csv` is the lower-risk threshold-only alternative.


In [ ]:
# 1. Imports and embedded V5 results
import base64, io, zipfile, json, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

BUNDLE_BASE64 = '''...'''
work = Path('/content/v6_inputs')
work.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_BASE64))) as zf:
    zf.extractall(work)
print('Loaded V5 bundle files:', sorted(str(p.relative_to(work)) for p in work.rglob('*') if p.is_file()))


In [ ]:
# 2. Load exact probabilities and labels
sel=np.load(work/'selected_predictions.npz',allow_pickle=True)
LABELS=sel['label_names'].tolist()
y=sel['labels'].astype(np.int8)
v2_valid=sel['valid'].astype(float)
v2_test=sel['test'].astype(float)
valid_ids=sel['valid_ids'].astype(str)
test_ids=sel['test_ids'].astype(str)
old_thresholds=sel['thresholds'].astype(float)

bb=np.load(next(work.glob('banglabert_v2style_*/predictions.npz')))
bb_valid_e4=bb['valid'][0].astype(float)
bb_test_e4=bb['test'][0].astype(float)
fold_frame=pd.read_csv(work/'validation_folds.csv',dtype={'ID':str})
fold_map=dict(zip(fold_frame.ID,fold_frame.fold))
fold_ids=np.array([fold_map[i] for i in valid_ids])
print('rows:',len(y),'validation /',len(test_ids),'test')
print('labels:',LABELS)


In [ ]:
# 3. Official competition metric
def official_components(target,pred):
    sa=f1_score(target[:,0],pred[:,0],average='macro',zero_division=0)
    attr=[f1_score(target[:,j],pred[:,j],average='binary',zero_division=0) for j in range(1,7)]
    sb=float(np.mean(attr))
    return {'final':0.5*sa+0.5*sb,'subtask_A':sa,'subtask_B':sb,'attribute_f1':dict(zip(LABELS[1:],attr))}

def objective_for_label(target,pred,j):
    return f1_score(target,pred,average='macro' if j==0 else 'binary',zero_division=0)

GRID=np.arange(0.02,0.805,0.005)
def best_threshold(target,probability,j):
    scores=np.array([objective_for_label(target,probability>=t,j) for t in GRID])
    best=np.flatnonzero(np.isclose(scores,scores.max(),atol=1e-12,rtol=0))
    # middle of a tied plateau is less brittle than choosing an edge
    k=best[len(best)//2]
    return float(GRID[k]),float(scores[k])

def fit_official_thresholds(target,probability):
    return np.array([best_threshold(target[:,j],probability[:,j],j)[0] for j in range(7)])

def crossfit_score(probability):
    pred=np.zeros_like(y,dtype=np.int8)
    fold_scores=[]
    for f in range(5):
        held=fold_ids==f
        t=fit_official_thresholds(y[~held],probability[~held])
        pred[held]=(probability[held]>=t).astype(np.int8)
        fold_scores.append(official_components(y[held],pred[held])['final'])
    return official_components(y,pred),fold_scores,pred

print('V5 thresholds evaluated with OFFICIAL metric:',official_components(y,(v2_valid>=old_thresholds).astype(np.int8)))


In [ ]:
# 4. Metric-corrected V2 baseline
metric_fix_thresholds=fit_official_thresholds(y,v2_valid)
metric_fix=official_components(y,(v2_valid>=metric_fix_thresholds).astype(np.int8))
metric_fix_cv,metric_fix_folds,_=crossfit_score(v2_valid)
print('thresholds:',dict(zip(LABELS,np.round(metric_fix_thresholds,3))))
print('full validation:',metric_fix)
print('grouped cross-fit:',metric_fix_cv)
print('fold scores:',np.round(metric_fix_folds,4))


In [ ]:
# 5. Conservative label-wise 5% BanglaBERT epoch-4 blend
BB_WEIGHTS=np.array([0.05,0.05,0.00,0.05,0.05,0.05,0.00])
blend_valid=(1-BB_WEIGHTS)*v2_valid + BB_WEIGHTS*bb_valid_e4
blend_test=(1-BB_WEIGHTS)*v2_test + BB_WEIGHTS*bb_test_e4
blend_thresholds=fit_official_thresholds(y,blend_valid)
blend_full=official_components(y,(blend_valid>=blend_thresholds).astype(np.int8))
blend_cv,blend_folds,_=crossfit_score(blend_valid)
print('weights:',dict(zip(LABELS,BB_WEIGHTS)))
print('thresholds:',dict(zip(LABELS,np.round(blend_thresholds,3))))
print('full validation:',blend_full)
print('grouped cross-fit:',blend_cv)
print('fold scores:',np.round(blend_folds,4))
print('cross-fit gain over metric-corrected V2:',blend_cv['final']-metric_fix_cv['final'])


In [ ]:
# 6. Generate submissions
def make_submission(probability,thresholds,path):
    pred=(probability>=thresholds).astype(np.int8)
    out=pd.DataFrame(pred,columns=LABELS)
    out.insert(0,'ID',test_ids)
    assert out.ID.is_unique and len(out)==len(test_ids)
    assert out[LABELS].isin([0,1]).all().all()
    out.to_csv(path,index=False)
    return out

metric_path='/content/submission_v6_metric_fix.csv'
recommended_path='/content/submission_v6_recommended.csv'
metric_sub=make_submission(v2_test,metric_fix_thresholds,metric_path)
recommended_sub=make_submission(blend_test,blend_thresholds,recommended_path)

old_pred=(v2_test>=old_thresholds).astype(np.int8)
new_pred=recommended_sub[LABELS].to_numpy()
print('Recommended positive counts:',recommended_sub[LABELS].sum().to_dict())
print('Changed cells vs V5:',dict(zip(LABELS,(new_pred!=old_pred).sum(0).tolist())))
print('Saved:',recommended_path,'and',metric_path)


In [ ]:
# 7. Save diagnostics and download the recommended submission
report={
    'version':'v6_metric_corrected_existing_predictions',
    'v5_official_score_on_validation_using_old_thresholds':official_components(y,(v2_valid>=old_thresholds).astype(np.int8)),
    'metric_fix_thresholds':dict(zip(LABELS,metric_fix_thresholds.tolist())),
    'metric_fix_full_validation':metric_fix,
    'metric_fix_crossfit':metric_fix_cv,
    'recommended_weights':dict(zip(LABELS,BB_WEIGHTS.tolist())),
    'recommended_thresholds':dict(zip(LABELS,blend_thresholds.tolist())),
    'recommended_full_validation':blend_full,
    'recommended_crossfit':blend_cv,
    'warning':'Validation diagnostics do not guarantee hidden-test/Kaggle performance.'
}
Path('/content/v6_tuning_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps(report,indent=2))
try:
    from google.colab import files
    files.download(recommended_path)
    files.download('/content/v6_tuning_report.json')
except Exception:
    pass



## What to submit
Submit **`submission_v6_recommended.csv` first**.

Why this candidate is preferred:
- It corrects the competition-metric mismatch in V5.
- It keeps the proven V2 predictions as 95–100% of the ensemble.
- The 5% BanglaBERT addition improved the grouped cross-fitted official score rather than only the fully tuned validation score.
- It avoids adding the weaker XLM-R candidate and avoids aggressive weight optimization.

The validation set is small, especially for `implicit_toxicity`, `threatening`, and `sexual_content`, so the hidden-test score can still move differently. Keep the prior 0.539 submission until Kaggle confirms V6 is better.
